# CynipsNet fine-tuning for the STM32N6570-DK (v4: punctuation, round 4)

Fine-tunes the speech model extracted from the board's firmware on Indian-accented English (Svarah),
recordings made **through the board's own microphone**, and LibriSpeech, and teaches it to put
**commas, full stops and question marks** in its text. It saves a float checkpoint; quantization,
the ST Edge AI compile and the firmware build run on your PC with `cynipsnet_re/deploy/deploy.py`.

**Before you start**
1. *Runtime → Change runtime type → GPU* (T4 is enough).
2. In Google Drive, the folder `MyDrive/cynipsnet/` must hold:
   * `cynipsnet_punct.pt`, the round-3 checkpoint (already there from round 3). `START_FROM` can also be
     `cynipsnet_finetuned.pt` (round 1, no punctuation yet) or `cynipsnet_v1_float.pt` (the original).
   * optional: recordings from `datacollect/karaoke_app.py`, zipped (`*.zip`, each with a
     `manifest.jsonl`). A zip whose name contains `test` is used **only for testing**.
   * optional: a Common Voice English release from **Mozilla Data Collective** (`*.tar.gz`).
3. `HF_TOKEN` in the Colab secrets (key icon), with the Svarah terms accepted on Hugging Face.

Punctuation is learned from **LibriSpeech-PC** (OpenSLR 145, CC BY 4.0: punctuated transcripts of the
LibriSpeech audio), downloaded below. Clips whose transcript has no punctuation (most likely Svarah's)
still train the words; there the model is not penalized for a mark.

Weights derived from CynipsNet are CC BY-NC-SA 4.0 (non-commercial).

In [ ]:
!nvidia-smi -L
!pip -q install datasets soundfile
import torch
assert torch.cuda.is_available(), "Switch the runtime to GPU first"

## 1. Training library (identical to `cynipsnet_re/colab/cynipsnet_finetune.py`, tested on the PC)

In [ ]:
%%writefile cynipsnet_finetune.py
"""Fine-tune CynipsNet (extracted from the STM32N6 firmware) on accented and board-microphone speech.

Everything the Colab notebook runs lives here, so the same code can be smoke-tested on a PC.
The front-end reproduces the firmware (src/asr_engine/mel, norm): 64 Slaney log-mels, 20 ms Hann
window, 10 ms hop, n_fft 512, pre-emphasis 0.97, per-band normalization, int8 input grid.
Output: a float checkpoint for cynipsnet_re/deploy (quantization and firmware build run locally,
where ST Edge AI is installed).
"""
import io
import json
import math
import os
import random
import re
import tarfile
import time
import zipfile
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

SR = 16000
TOKENS = "_ ABCDEFGHIJKLMNOPQRSTUVWXYZ'"  # the original 29 classes
PUNCT = ",.?"  # classes 29-31 of a model trained with punctuation (add_punctuation)
TOKENS_PUNCT = TOKENS + PUNCT
TOKEN_ID = {c: i for i, c in enumerate(TOKENS_PUNCT)}
INPUT_SCALE = 0.0628561676
MAX_SECONDS = 20.0

# ----------------------------------------------------------------------------- model (cynipsnet_arch.py)
BLOCKS = [(256, 33, 5), (256, 39, 5), (256, 39, 5), (512, 51, 5), (512, 63, 5), (512, 75, 5)]


class TFSameConv1d(nn.Conv1d):
    """Strided conv with TensorFlow SAME padding, as the deployed NPU graph computes C1: for an even
    number of frames 15 frames on the left and 16 on the right (PyTorch's symmetric 16/16 would shift
    the output by one frame). Parameter names are those of nn.Conv1d."""

    def forward(self, x):
        k, s, t = self.kernel_size[0], self.stride[0], x.shape[-1]
        total = max((-(-t // s) - 1) * s + k - t, 0)
        return F.conv1d(F.pad(x, (total // 2, total - total // 2)), self.weight, self.bias, s, 0,
                        self.dilation, self.groups)


class SeparableConv(nn.Module):
    def __init__(self, cin, cout, kernel, stride=1, dilation=1):
        super().__init__()
        pad = dilation * (kernel - 1) // 2
        if stride > 1:
            self.dw = TFSameConv1d(cin, cin, kernel, stride=stride, groups=cin)
        else:
            self.dw = nn.Conv1d(cin, cin, kernel, padding=pad, dilation=dilation, groups=cin)
        self.pw = nn.Conv1d(cin, cout, 1)

    def forward(self, x):
        return self.pw(self.dw(x))


class Block(nn.Module):
    def __init__(self, cin, cout, kernel, repeats):
        super().__init__()
        self.subs = nn.ModuleList(SeparableConv(cin if i == 0 else cout, cout, kernel) for i in range(repeats))
        self.residual = nn.Conv1d(cin, cout, 1)

    def forward(self, x):
        y = x
        for i, sub in enumerate(self.subs):
            y = sub(y)
            if i < len(self.subs) - 1:
                y = torch.relu(y)
        return torch.relu(y + self.residual(x))


class CynipsNet(nn.Module):
    def __init__(self, n_classes=len(TOKENS)):
        super().__init__()
        self.c1 = SeparableConv(64, 256, 33, stride=2)
        blocks, cin = [], 256
        for cout, kernel, repeats in BLOCKS:
            blocks.append(Block(cin, cout, kernel, repeats))
            cin = cout
        self.blocks = nn.Sequential(*blocks)
        self.c2 = SeparableConv(512, 512, 87, dilation=2)
        self.c3 = nn.Conv1d(512, 1024, 1)
        self.c4 = nn.Conv1d(1024, n_classes, 1)

    def forward(self, mel):
        x = torch.relu(self.c1(mel))
        x = self.blocks(x)
        x = torch.relu(self.c2(x))
        x = torch.relu(self.c3(x))
        return self.c4(x)


def load_baseline(path, device):
    """Load the extracted checkpoint; refuse to continue from random weights."""
    if not os.path.exists(path):
        raise FileNotFoundError("%s not found: upload cynipsnet_v1_float.pt first. Training from random "
                                "weights produces a model that outputs nothing." % path)
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    state = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    model = CynipsNet(n_classes=state["c4.weight"].shape[0])  # 29, or 32 with punctuation
    model.load_state_dict(state, strict=True)
    return model.to(device)


def add_punctuation(model):
    """Grow the output layer from 29 to 32 classes (comma, full stop, question mark).

    The 29 trained rows are kept. A mark ends a word where a space would start the next one, so the
    new rows start as the space row, half a logit less likely. That is close enough for training to
    move them within a few hundred steps (Adam moves a weight by about lr per step, so a gap of several
    logits takes thousands), and harmless meanwhile: a space always follows a mark (format_text), so a
    mark printed instead of a space costs no word."""
    old = model.c4
    if old.out_channels == len(TOKENS_PUNCT):
        return model
    new = nn.Conv1d(old.in_channels, len(TOKENS_PUNCT), 1).to(old.weight.device)
    space = TOKEN_ID[" "]
    with torch.no_grad():
        new.weight[:len(TOKENS)] = old.weight
        new.bias[:len(TOKENS)] = old.bias
        for k in range(len(TOKENS), len(TOKENS_PUNCT)):
            row = old.weight[space]
            new.weight[k] = row + 0.01 * row.std() * torch.randn_like(row)
            new.bias[k] = old.bias[space] - 0.5
    model.c4 = new
    return model


# ----------------------------------------------------------------------------- front-end (firmware mel.c / norm.c)
def _mel_bank():
    def hz_to_mel(hz):
        return np.where(hz < 1000.0, hz / (200.0 / 3.0), 15.0 + np.log(np.maximum(hz, 1e-12) / 1000.0) / (np.log(6.4) / 27.0))

    def mel_to_hz(m):
        return np.where(m < 15.0, m * (200.0 / 3.0), 1000.0 * np.exp((np.log(6.4) / 27.0) * (m - 15.0)))

    edges = mel_to_hz(np.linspace(hz_to_mel(0.0), hz_to_mel(8000.0), 66))
    bins = np.arange(257) * (8000.0 / 256)
    bank = np.zeros((64, 257))
    for b in range(64):
        lo, c, hi = edges[b], edges[b + 1], edges[b + 2]
        w = np.minimum((bins - lo) / (c - lo), (hi - bins) / (hi - c))
        bank[b] = np.where(w > 0, w * 2.0 / (hi - lo), 0.0)
    return torch.tensor(bank, dtype=torch.float32)


class FrontEnd(nn.Module):
    """Batched waveform -> int8-grid normalized log-mel, plus valid frame counts."""

    def __init__(self):
        super().__init__()
        self.register_buffer("bank", _mel_bank())
        n = torch.arange(320, dtype=torch.float64)
        self.register_buffer("window", (0.5 - 0.5 * torch.cos(2 * math.pi * n / 319)).float())

    def forward(self, wave, lengths):
        """wave: [B, N] float in [-1, 1) (PCM16 / 32768); lengths: samples. -> ([B, 64, T], frames)."""
        e = torch.cat([wave[:, :1], wave[:, 1:] - 0.97 * wave[:, :-1]], dim=1)
        frames = e.unfold(1, 320, 160) * self.window
        power = torch.fft.rfft(frames, n=512, dim=-1).abs().pow(2)
        logmel = torch.log(power @ self.bank.T + 2.0 ** -24).transpose(1, 2)  # [B, 64, T]
        nframes = torch.clamp((lengths - 320) // 160 + 1, min=1)
        t = torch.arange(logmel.shape[2], device=wave.device)[None, None, :]
        mask = (t < nframes[:, None, None]).float()
        count = nframes[:, None, None].float()
        mu = (logmel * mask).sum(2, keepdim=True) / count
        var = ((logmel - mu) ** 2 * mask).sum(2, keepdim=True) / count
        x = (logmel - mu) / torch.clamp(var.sqrt(), min=1e-6)
        # The NPU gets int8 codes of x / INPUT_SCALE: train on that grid (straight-through gradient).
        xq = torch.clamp(torch.round(x / INPUT_SCALE), -128, 127) * INPUT_SCALE
        x = x + (xq - x).detach()
        return x * mask, nframes


# ----------------------------------------------------------------------------- text
_ONES = ("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen "
         "fifteen sixteen seventeen eighteen nineteen").split()
_TENS = "_ _ twenty thirty forty fifty sixty seventy eighty ninety".split()


def number_words(n):
    """Integer to English words, the way the board's alphabet spells numbers (no digits)."""
    if n < 20:
        return _ONES[n]
    if n < 100:
        return _TENS[n // 10] + ("" if n % 10 == 0 else " " + _ONES[n % 10])
    if n < 1000:
        rest = n % 100
        return _ONES[n // 100] + " hundred" + ("" if rest == 0 else " and " + number_words(rest))
    for value, name in ((10 ** 9, "billion"), (10 ** 6, "million"), (1000, "thousand")):
        if n >= value:
            rest = n % value
            return number_words(n // value) + " " + name + ("" if rest == 0 else
                                                           (" and " if rest < 100 else " ") + number_words(rest))
    return str(n)


def ordinal_words(n):
    words = number_words(n).split()
    last = words[-1]
    special = dict(one="first", two="second", three="third", five="fifth", eight="eighth", nine="ninth",
                   twelve="twelfth")
    words[-1] = special.get(last, last[:-1] + "ieth" if last.endswith("y") else last + "th")
    return " ".join(words)


_ABBREVIATIONS = [(r"\bMrs\.", "Missus"), (r"\bMr\.", "Mister"), (r"\bMs\.", "Miss"), (r"\bDr\.", "Doctor"),
                  (r"\bvs\.", "versus"), (r"\betc\.", "etcetera"), (r"\be\.g\.", "for example"),
                  (r"\bi\.e\.", "that is")]


def has_marks(raw):
    """Whether a raw transcript is punctuated: a mark followed by a space or the end (not 3.5)."""
    return re.search(r"[.,?!;:](\s|$)", raw) is not None


def format_text(text):
    """Punctuation as the board prints it: a mark sticks to the word before it and is followed by one
    space; nothing before the first word; a run of marks keeps the strongest (? over . over ,)."""
    text = re.sub(r"\s+([,.?])", r"\1", " " + text)
    text = re.sub(r"[,.?]+", lambda m: "?" if "?" in m.group() else ("." if "." in m.group() else ","), text)
    text = re.sub(r"^[\s,.?]+", "", text)
    text = re.sub(r"([,.?])(?=\S)", r"\1 ", text)
    return " ".join(text.split())


def strip_punct(text):
    """Words only, for the WER: punctuation is scored separately (punct_report)."""
    return " ".join(re.sub(r"[,.?]", " ", text).split())


def normalize_text(text, punct=False):
    """Model alphabet: A-Z, apostrophe, space; numbers spelled out (number_words).

    punct=True also keeps comma, full stop and question mark (! becomes ., ; and : become ,), with
    abbreviations and number punctuation (Mr., 3.5, 1,000, 7:30) spelled out first."""
    text = text.replace("’", "'").replace("‘", "'")
    import unicodedata
    # Fold accents (an accented e -> e); any other non-ASCII mark (dashes, quotes) separates words.
    text = "".join(c if ord(c) < 128 else " " for c in unicodedata.normalize("NFKD", text)
                   if not unicodedata.combining(c))
    if punct:
        for pattern, words in _ABBREVIATIONS:
            text = re.sub(pattern, words, text, flags=re.IGNORECASE)
        text = re.sub(r"(\d),(?=\d{3}\b)", r"\1", text)
        text = re.sub(r"(\d)\.(\d)", r"\1 point \2", text)
        text = re.sub(r"(\d):(\d)", r"\1 \2", text)
        text = text.replace("!", ".").replace(";", ",").replace(":", ",")
    if re.search(r"\d", text):
        text = re.sub(r"(\d+)(st|nd|rd|th)\b", lambda m: " " + ordinal_words(int(m.group(1))) + " ", text,
                      flags=re.IGNORECASE)
        text = re.sub(r"\d+", lambda m: " " + number_words(int(m.group())) + " ", text)
    text = text.upper().replace("-", " ")
    text = re.sub(r"[^A-Z' ,.?]+" if punct else r"[^A-Z' ]+", " ", text)
    # An apostrophe is a letter only inside a word (IT'S); quote marks around words are not.
    text = re.sub(r"(?<![A-Z])'|'(?![A-Z])", " ", text)
    text = format_text(text) if punct else " ".join(text.split())
    return text if re.search(r"[A-Z]", text) else None


def encode(text):
    return [TOKEN_ID[c] for c in text]


def ctc_target(text):
    """The class sequence a clip is trained on. A mark takes the place of the space after it
    ("SIZE,IN" for "SIZE, IN"): the network then chooses mark or space at the one step where it
    already ends a word, instead of having to emit two symbols there, and both decoders print a
    space after every mark anyway."""
    return encode(re.sub(r"([,.?]) ", r"\1", text))


def _split_marks(text):
    """'HELLO, WORLD.' -> [('HELLO', ','), ('WORLD', '.')]."""
    out = []
    for token in text.split():
        word, mark = token.rstrip(PUNCT), token[-1] if token[-1] in PUNCT else ""
        if word:
            out.append((word, mark))
        elif mark and out and not out[-1][1]:
            out[-1] = (out[-1][0], mark)
    return out


def punct_report(pairs):
    """Precision, recall and F1 (%) of each mark over (reference, hypothesis) pairs.

    The words are aligned by edit distance; a mark counts as right when it follows the aligned word
    (right or misrecognized) in both. {'all': (P, R, F1), ',': ..., '.': ..., '?': ...}"""
    counts = {m: [0, 0, 0] for m in PUNCT}  # true positives, false positives, false negatives
    for ref, hyp in pairs:
        r, h = _split_marks(ref), _split_marks(hyp)
        d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
        for i in range(len(r) + 1):
            d[i][0] = i
        for j in range(len(h) + 1):
            d[0][j] = j
        for i in range(1, len(r) + 1):
            for j in range(1, len(h) + 1):
                d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + (r[i - 1][0] != h[j - 1][0]))
        i, j, links = len(r), len(h), []
        while i > 0 or j > 0:
            if i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + (r[i - 1][0] != h[j - 1][0]):
                links.append((r[i - 1][1], h[j - 1][1]))
                i, j = i - 1, j - 1
            elif i > 0 and d[i][j] == d[i - 1][j] + 1:
                links.append((r[i - 1][1], ""))
                i -= 1
            else:
                links.append(("", h[j - 1][1]))
                j -= 1
        for rm, hm in links:
            if rm and rm == hm:
                counts[rm][0] += 1
                continue
            if hm:
                counts[hm][1] += 1
            if rm:
                counts[rm][2] += 1

    def prf(tp, fp, fn):
        p = 100.0 * tp / max(1, tp + fp)
        r = 100.0 * tp / max(1, tp + fn)
        return round(p, 1), round(r, 1), round(2 * p * r / max(1e-9, p + r), 1)
    total = [sum(c[k] for c in counts.values()) for k in range(3)]
    return dict(all=prf(*total), **{m: prf(*c) for m, c in counts.items()})


# ----------------------------------------------------------------------------- data sources
class Item:
    """One clip. punct: its text is punctuated, so a model with punctuation classes is trained on the
    marks; otherwise they are free (see train)."""
    __slots__ = ("pcm", "text", "speaker", "source", "punct")

    def __init__(self, pcm, text, speaker, source, punct=False):
        self.pcm, self.text, self.speaker, self.source, self.punct = pcm, text, speaker, source, punct


def to_pcm16(audio, sr):
    """float or int array at sr -> int16 at 16 kHz."""
    audio = np.asarray(audio)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if audio.dtype != np.int16:
        audio = np.clip(audio.astype(np.float32), -1.0, 1.0)
    else:
        audio = audio.astype(np.float32) / 32768.0
    if sr != SR:
        from math import gcd
        from scipy.signal import resample_poly
        g = gcd(int(sr), SR)
        audio = resample_poly(audio, SR // g, int(sr) // g).astype(np.float32)
    return np.clip(np.round(audio * 32767.0), -32768, 32767).astype(np.int16)


def _keep(pcm, text):
    return text is not None and 0.5 <= len(pcm) / SR <= MAX_SECONDS


def _text(raw, punct, trusted=False):
    """(normalized text, punctuated?) of a raw transcript. trusted: a punctuated source (LibriSpeech-PC),
    where a segment without marks really has none; otherwise unmarked text counts as unpunctuated."""
    if punct and (trusted or has_marks(raw)):
        return normalize_text(raw, punct=True), True
    return normalize_text(raw), False


def load_librispeech_pc(manifest):
    """LibriSpeech-PC (OpenSLR 145, CC BY 4.0): {utterance id: punctuated text} from one of its .json
    manifests, e.g. train-clean-100.json. The ids match LibriSpeech's (1272-128104-0000)."""
    out = {}
    for line in open(manifest, encoding="utf-8"):
        row = json.loads(line)
        out[os.path.basename(row["audio_filepath"]).rsplit(".", 1)[0]] = row["text"]
    return out


def load_librispeech(root, limit=None, seed=0, source="librispeech", punct_text=None):
    """A LibriSpeech-style directory (speaker/chapter/*.flac + *.trans.txt).

    punct_text: {id: text} from load_librispeech_pc; then only utterances it covers are used, with
    their punctuated text."""
    import soundfile as sf
    rows = []
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f.endswith(".trans.txt"):
                for line in open(os.path.join(dirpath, f), encoding="utf-8"):
                    uid, text = line.strip().split(" ", 1)
                    if punct_text is None or uid in punct_text:
                        rows.append((os.path.join(dirpath, uid + ".flac"),
                                     text if punct_text is None else punct_text[uid], uid.split("-")[0]))
    rows.sort()
    random.Random(seed).shuffle(rows)
    items = []
    for path, raw, spk in rows[:limit] if limit else rows:
        pcm, sr = sf.read(path, dtype="int16")
        text, marked = _text(raw, punct_text is not None, trusted=True)
        if _keep(pcm, text):
            items.append(Item(to_pcm16(pcm, sr), text, "ls_" + spk, source, marked))
    return items


def load_librispeech_hf(split="train.100", shards=(0,), limit=None, seed=0, source="librispeech",
                        punct_text=None):
    """LibriSpeech "clean" from its Hugging Face copy (openslr/librispeech_asr, parquet shards).

    On Colab this downloads in seconds, against tens of minutes from openslr.org, and only the
    shards asked for are fetched. split: "train.100" (shards 0-13, about 2000 utterances each),
    "validation" (dev-clean, shard 0) or "test" (test-clean, shard 0). limit is spread over the shards.
    punct_text: as in load_librispeech."""
    import pyarrow.parquet as pq
    import soundfile as sf
    from huggingface_hub import hf_hub_download
    shards = list(shards)
    per_shard = -(-limit // len(shards)) if limit else None
    rng = random.Random(seed)
    items = []
    for shard in shards:
        path = hf_hub_download("openslr/librispeech_asr", "clean/%s/%04d.parquet" % (split, shard),
                               repo_type="dataset")
        table = pq.read_table(path, columns=["audio", "text", "speaker_id", "id"])
        audio, texts, speakers, ids = (table.column(c) for c in ("audio", "text", "speaker_id", "id"))
        order = [i for i in range(table.num_rows) if punct_text is None or ids[i].as_py() in punct_text]
        rng.shuffle(order)
        for i in order[:per_shard] if per_shard else order:
            pcm, sr = sf.read(io.BytesIO(audio[i].as_py()["bytes"]), dtype="int16")
            raw = texts[i].as_py() if punct_text is None else punct_text[ids[i].as_py()]
            text, marked = _text(raw, punct_text is not None, trusted=True)
            if _keep(pcm, text):
                items.append(Item(to_pcm16(pcm, sr), text, "ls_%d" % speakers[i].as_py(), source, marked))
        del table, audio, texts, speakers, ids
    return items[:limit] if limit else items


def load_svarah(token, limit=None, punct=False):
    """ai4bharat/Svarah (CC BY 4.0, gated: accept its terms on Hugging Face and pass a read token).

    There is no speaker column; speaker identity is approximated by the metadata fingerprint.
    punct: keep the marks of clips whose transcript has them."""
    import soundfile as sf
    from datasets import Audio, load_dataset
    ds = load_dataset("ai4bharat/Svarah", split="test", token=token)
    ds = ds.cast_column("audio_filepath", Audio(decode=False))
    meta = ["gender", "age-group", "primary_language", "native_place_state", "native_place_district",
            "highest_qualification", "job_category", "occupation_domain"]
    items = []
    for i, row in enumerate(ds):
        if limit and i >= limit:
            break
        audio = row["audio_filepath"]
        data = audio.get("bytes")
        pcm, sr = sf.read(io.BytesIO(data) if data else audio["path"], dtype="float32")
        text, marked = _text(row["text"], punct)
        pcm = to_pcm16(pcm, sr)
        if _keep(pcm, text):
            spk = "svarah_" + "|".join(str(row.get(k, "")) for k in meta)
            items.append(Item(pcm, text, spk, "svarah", marked))
    return items


def load_common_voice(cv_dir, accent_regex=r"india|south asia", splits=("train", "dev", "test"), limit=None,
                      punct=False):
    """A Common Voice English release from Mozilla Data Collective, extracted: <cv_dir>/{clips,*.tsv}.

    Keeps clips whose self-reported accent matches accent_regex. Returns {split: [Item]} using
    Common Voice's own speaker-disjoint splits. punct: keep the sentences' marks."""
    import csv
    out = {}
    for split in splits:
        tsv = os.path.join(cv_dir, split + ".tsv")
        if not os.path.exists(tsv):
            continue
        items = []
        with open(tsv, encoding="utf-8") as f:
            for row in csv.DictReader(f, delimiter="\t", quoting=csv.QUOTE_NONE):
                accent = (row.get("accents") or row.get("accent") or "").lower()
                if not re.search(accent_regex, accent):
                    continue
                text, marked = _text(row["sentence"], punct)
                if text is None:
                    continue
                import torchaudio
                wav, sr = torchaudio.load(os.path.join(cv_dir, "clips", row["path"]))
                pcm = to_pcm16(wav.mean(0).numpy(), sr)
                if _keep(pcm, text):
                    items.append(Item(pcm, text, "cv_" + row["client_id"], "commonvoice", marked))
                if limit and len(items) >= limit:
                    break
        out[split] = items
    return out


def load_manifest(path, source=None, punct=False):
    """manifest.jsonl from datacollect/board_record.py (audio path relative to the manifest).

    punct: use the prompt as written (raw_text) when it is punctuated."""
    import soundfile as sf
    base = os.path.dirname(os.path.abspath(path))
    items = []
    for line in open(path, encoding="utf-8"):
        row = json.loads(line)
        pcm, sr = sf.read(os.path.join(base, row["audio"]), dtype="int16")
        text, marked = _text(row.get("raw_text") or row["text"], punct)
        pcm = to_pcm16(pcm, sr)
        if _keep(pcm, text):
            items.append(Item(pcm, text, "rec_" + row.get("speaker", "unknown"), source or row.get("source", "rec"),
                              marked))
    return items


def split_by_speaker(items, dev=0.1, test=0.2, seed=0):
    """Speaker-disjoint train/dev/test; with a single speaker, split that speaker's takes instead."""
    speakers = sorted({it.speaker for it in items})
    if len(speakers) < 3:
        rng = random.Random(seed)
        idx = list(range(len(items)))
        rng.shuffle(idx)
        n_test, n_dev = int(len(idx) * test), int(len(idx) * dev)
        pick = lambda ids: [items[i] for i in ids]
        return pick(idx[n_test + n_dev:]), pick(idx[n_test:n_test + n_dev]), pick(idx[:n_test])
    rng = random.Random(seed)
    rng.shuffle(speakers)
    n_test = max(1, int(round(len(speakers) * test)))
    n_dev = max(1, int(round(len(speakers) * dev)))
    test_s, dev_s = set(speakers[:n_test]), set(speakers[n_test:n_test + n_dev])
    tr = [it for it in items if it.speaker not in test_s | dev_s]
    return tr, [it for it in items if it.speaker in dev_s], [it for it in items if it.speaker in test_s]


def describe(name, items):
    hours = sum(len(it.pcm) for it in items) / SR / 3600
    return "%-24s %6d utterances %6.2f h %4d speakers" % (name, len(items), hours, len({i.speaker for i in items}))


# ----------------------------------------------------------------------------- augmentation (batched, on device)
def _synthetic_rir(rt60, device):
    n = int(SR * min(rt60 * 1.2, 1.0))
    t = torch.arange(n, device=device) / SR
    rir = torch.randn(n, device=device) * torch.exp(-6.9 * t / rt60)
    rir[0] = 1.0
    return rir / rir.norm()


def _fft_size(n):
    """Power-of-two FFT length >= n. Every FFT here uses one: at arbitrary clip lengths cuFFT builds a
    plan per length (Bluestein for large prime factors) and fails with CUFFT_INTERNAL_ERROR on Colab."""
    return 1 << (int(n) - 1).bit_length()


def augment(wave, lengths, p_noise=0.7, p_reverb=0.4, p_eq=0.5):
    """Gain, colored noise at 5-35 dB SNR, synthetic reverberation, mild band limiting: the far-field,
    small-MEMS-mic conditions the board sees."""
    b, n = wave.shape
    dev = wave.device
    out = []
    for i in range(b):
        x = wave[i, :lengths[i]].float()
        length = x.numel()
        if random.random() < p_reverb:
            rir = _synthetic_rir(random.uniform(0.15, 0.8), dev)
            nfft = _fft_size(length + rir.numel() - 1)
            y = torch.fft.irfft(torch.fft.rfft(x, nfft) * torch.fft.rfft(rir, nfft), nfft)[:length]
            x = y * (x.norm() / (y.norm() + 1e-8))
        if random.random() < p_eq:
            # Small-MEMS-mic band: smooth high-pass and low-pass in the frequency domain.
            nfft = _fft_size(length)
            spec = torch.fft.rfft(x, nfft)
            f = torch.linspace(0, SR / 2, spec.numel(), device=dev)
            hp, lp = random.uniform(60, 250), random.uniform(3500, 7500)
            gain = torch.sigmoid((f - hp) / (hp / 4)) * torch.sigmoid((lp - f) / 400.0)
            x = torch.fft.irfft(spec * gain, nfft)[:length]
        if random.random() < p_noise:
            nfft = _fft_size(length)
            noise = torch.randn(nfft, device=dev)
            alpha = random.choice([0.0, 0.5, 1.0])  # white, between, pink-ish
            if alpha > 0:
                spec = torch.fft.rfft(noise)
                f = torch.arange(spec.numel(), device=dev).clamp(min=1).float()
                noise = torch.fft.irfft(spec / f ** (alpha / 2), nfft)
            noise = noise[:length]
            snr = random.uniform(5, 35)
            noise = noise * (x.norm() / (noise.norm() + 1e-8)) * 10 ** (-snr / 20)
            x = x + noise
        x = x * 10 ** (random.uniform(-12, 6) / 20)
        out.append(torch.clamp(x, -1.0, 1.0))
    res = torch.zeros_like(wave)
    for i, x in enumerate(out):
        res[i, :x.numel()] = x
    return res


def spec_augment(x, frames, n_freq=2, f_width=10, n_time=2, t_width=40):
    for i in range(x.shape[0]):
        for _ in range(n_freq):
            w = random.randint(0, f_width)
            f0 = random.randint(0, 64 - w)
            x[i, f0:f0 + w, :] = 0
        T = int(frames[i])
        for _ in range(n_time):
            w = random.randint(0, min(t_width, max(1, T // 10)))
            t0 = random.randint(0, max(0, T - w))
            x[i, :, t0:t0 + w] = 0
    return x


# ----------------------------------------------------------------------------- batching, loss, decoding
def batches(items, max_seconds=240.0, shuffle=True, seed=0):
    """Length-sorted buckets so padding stays small; a batch holds about max_seconds of audio."""
    order = sorted(range(len(items)), key=lambda i: len(items[i].pcm))
    groups, cur, longest = [], [], 0
    for i in order:
        longest = max(longest, len(items[i].pcm))
        if cur and longest * (len(cur) + 1) / SR > max_seconds:
            groups.append(cur)
            cur, longest = [], len(items[i].pcm)
        cur.append(i)
    if cur:
        groups.append(cur)
    if shuffle:
        random.Random(seed).shuffle(groups)
    return groups


EVAL_PAD = (0.5, 1.0)  # seconds of noise floor before / after a clip when evaluating


def noise_floor(pcm):
    """A clip's noise floor: 10th-percentile 20 ms RMS in int16 units (frontend.pad_with_noise_floor)."""
    n = len(pcm) // 320
    if n == 0:
        return 1.0
    rms = np.sqrt((pcm[:n * 320].astype(np.float64).reshape(n, 320) ** 2).mean(1))
    return max(float(np.percentile(rms, 10)), 1.0)


def join_clips(items, rng, gap=(0.15, 1.2)):
    """Several clips as one, with a pause of noise floor between them: speech goes on after a
    sentence ends, as in the board's stream. Without such clips a model learns to put a full stop
    where the audio ends, which on the board is the edge of every window, never the final text."""
    parts, level = [], noise_floor(items[0].pcm)
    for k, it in enumerate(items):
        if k:
            pause = rng.normal(0, level, int(rng.uniform(*gap) * SR))
            parts.append(np.clip(pause, -32768, 32767).astype(np.int16))
        parts.append(it.pcm)
    return Item(np.concatenate(parts), " ".join(it.text for it in items), items[0].speaker, items[0].source,
                items[0].punct)


def joined_pool(pool, fraction, rng, max_seconds=30.0):
    """The pool with `fraction` of its clips joined two or three at a time (join_clips); clips are only
    joined with clips of the same punctuation flag, and the audio per epoch stays the same."""
    if fraction <= 0:
        return pool
    order = rng.permutation(len(pool)).tolist()
    take = order[:int(len(pool) * fraction)]
    chosen = set(take)
    out = [pool[i] for i in range(len(pool)) if i not in chosen]
    for marked in (True, False):
        group = [pool[i] for i in take if pool[i].punct == marked]
        k = 0
        while k < len(group):
            want, chunk = int(rng.integers(2, 4)), [group[k]]
            k += 1
            while k < len(group) and len(chunk) < want and \
                    sum(len(c.pcm) for c in chunk) + len(group[k].pcm) <= max_seconds * SR:
                chunk.append(group[k])
                k += 1
            out.append(join_clips(chunk, rng) if len(chunk) > 1 else chunk[0])
    return out


def collate(items, idx, device, pad=None, seed=0):
    """Batch of clips. pad: None, "eval" (EVAL_PAD) or "train" (0.1-1 s before, 0.3-2.5 s after, random):
    the clip is surrounded by noise at its own noise floor, the way the board hears speech inside a
    live stream. Without it a word cut tight at its last sound tends to lose that sound."""
    rng = np.random.default_rng(seed)
    pcms = []
    for i in idx:
        p = items[i].pcm.astype(np.float32)
        if pad:
            before, after = EVAL_PAD if pad == "eval" else (rng.uniform(0.1, 1.0), rng.uniform(0.3, 2.5))
            level = noise_floor(items[i].pcm)
            p = np.concatenate([rng.normal(0, level, int(before * SR)), p, rng.normal(0, level, int(after * SR))])
        pcms.append(np.clip(p, -32768, 32767).astype(np.float32) / 32768.0)
    n = max(len(p) for p in pcms)
    wave = torch.zeros(len(idx), n)
    for k, p in enumerate(pcms):
        wave[k, :len(p)] = torch.from_numpy(p)
    lengths = torch.tensor([len(p) for p in pcms])
    targets = [torch.tensor(ctc_target(items[i].text)) for i in idx]
    return wave.to(device), lengths.to(device), targets


def out_lengths(frames):
    return (frames + 1) // 2  # C1 stride 2 with same padding


def greedy(logits):
    """logits [classes, T] -> text (collapse repeats, drop blanks; marks placed as format_text does)."""
    tokens = TOKENS_PUNCT[:logits.shape[0]]
    ids = logits.argmax(0).tolist()
    out, prev = [], -1
    for i in ids:
        if i != prev and i != 0:
            out.append(tokens[i])
        prev = i
    text = "".join(out)
    return format_text(text) if len(tokens) > len(TOKENS) else " ".join(text.split())


def wer(refs, hyps):
    """Word error rate (%) on the words only; punctuation is scored by punct_report."""
    def dist(r, h):
        d = list(range(len(h) + 1))
        for i in range(1, len(r) + 1):
            prev, d[0] = d[0], i
            for j in range(1, len(h) + 1):
                cur = min(d[j] + 1, d[j - 1] + 1, prev + (r[i - 1] != h[j - 1]))
                prev, d[j] = d[j], cur
        return d[len(h)]
    refs, hyps = [strip_punct(r).split() for r in refs], [strip_punct(h).split() for h in hyps]
    errs = sum(dist(r, h) for r, h in zip(refs, hyps))
    return 100.0 * errs / max(1, sum(len(r) for r in refs))


@torch.no_grad()
def evaluate(model, frontend, items, device, max_seconds=400.0, pad="eval"):
    """(WER %, [(reference, hypothesis)] in the order of items)."""
    model.eval()
    hyps = [""] * len(items)
    for b, idx in enumerate(batches(items, max_seconds, shuffle=False)):
        wave, lengths, _ = collate(items, idx, device, pad=pad, seed=b)
        x, frames = frontend(wave, lengths)
        logits = model(x)
        olen = out_lengths(frames)
        for k, i in enumerate(idx):
            hyps[i] = greedy(logits[k, :, :olen[k]])
    refs = [it.text for it in items]
    return wer(refs, hyps), list(zip(refs, hyps))


def train(model, frontend, train_sets, dev_sets, device, epochs=12, lr=1e-4, max_seconds=240.0,
          weights=None, guard_set=None, guard_margin=1.5, log=print, out_path="cynipsnet_finetuned.pt",
          max_steps=None, punct_weight=0.25, pad=True, head_lr_mult=None, join=0.0, question_boost=1.0):
    """Fine-tune on a weighted mixture of train_sets ({name: [Item]}); select on the dev sets.

    weights: {name: sampling multiplier} (e.g. board recordings x3). guard_set: a (name, items) dev
    set, normally LibriSpeech, whose WER may rise by at most guard_margin points over the start.
    A model with punctuation classes (add_punctuation) learns the marks from punctuated clips; on the
    others blank and marks are merged into one "nothing here" class, so a mark is never penalized
    where the transcript simply has none. Selection: mean dev WER + punct_weight x (100 - mark F1).
    pad: surround every training clip with noise floor (collate), as the board hears it.
    head_lr_mult: learning-rate factor of the output layer; 5 by default when it has new punctuation
    classes, which start untrained, else 1.
    join: fraction of the clips joined two or three at a time each epoch (joined_pool), so the model
    sees sentences followed by more speech. question_boost: how many times punctuated clips with a
    question mark are seen (questions are about 3 % of LibriSpeech-PC's marks)."""
    weights = weights or {}
    n_classes = model.c4.out_channels
    punct_model = n_classes > len(TOKENS)
    pool = []
    for name, items in train_sets.items():
        reps = weights.get(name, 1.0)
        full, frac = int(reps), reps - int(reps)
        pool += items * full + random.sample(items, int(len(items) * frac))
    if not punct_model and any(it.punct for it in pool):
        raise ValueError("punctuated clips need a model with punctuation classes: add_punctuation(model)")
    if punct_model and question_boost > 1:
        questions = [it for it in pool if it.punct and "?" in it.text]
        pool += questions * (int(round(question_boost)) - 1)
    log("training pool: %d utterances, %.1f h per epoch%s" % (
        len(pool), sum(len(i.pcm) for i in pool) / SR / 3600,
        ", %d with punctuation" % sum(it.punct for it in pool) if punct_model else ""))
    steps_per_epoch = len(batches(pool, max_seconds))
    total = steps_per_epoch * epochs
    head = list(model.c4.parameters())
    body = [p for p in model.parameters() if all(p is not h for h in head)]
    mult = head_lr_mult if head_lr_mult is not None else (5.0 if punct_model else 1.0)
    opt = torch.optim.AdamW([dict(params=body, lr=lr), dict(params=head, lr=lr * mult)], weight_decay=1e-3)
    warm = min(500, total // 10 + 1)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(s, total) / total)))
    ctc = nn.CTCLoss(blank=0, reduction="none", zero_infinity=True)
    nothing = [0] + list(range(len(TOKENS), n_classes))  # blank and the marks
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    def ctc_loss(logp, items, idx, targets, olen):
        """Mean over the batch of CTC / target length (nn.CTCLoss 'mean'), marks free where unpunctuated."""
        per = torch.zeros(len(idx), device=logp.device)
        for marked in (True, False):
            ks = [k for k, i in enumerate(idx) if (punct_model and items[i].punct) == marked]
            if not ks:
                continue
            sel = torch.tensor(ks, device=logp.device)
            lp = logp[sel]
            if punct_model and not marked:
                lp = torch.cat([torch.logsumexp(lp[:, nothing], 1, keepdim=True), lp[:, 1:len(TOKENS)]], 1)
            t = [targets[k] for k in ks]
            tl = torch.tensor([len(x) for x in t], device=logp.device)
            per[sel] = ctc(lp.permute(2, 0, 1), torch.cat(t).to(logp.device), olen[sel], tl) / tl.clamp(min=1)
        return per.mean()

    def dev_report():
        res, f1 = {}, []
        for name, items in list(dev_sets.items()) + ([guard_set] if guard_set else []):
            w, pairs = evaluate(model, frontend, items, device)
            res[name] = w
            marked = [p for p, it in zip(pairs, items) if it.punct]
            if punct_model and marked:
                res[name + " marks F1"] = punct_report(marked)["all"][2]
                if name in dev_sets:
                    f1.append(res[name + " marks F1"])
        wers = [res[n] for n in dev_sets]
        score = (float(np.mean(wers)) if wers else 0.0) + (punct_weight * (100.0 - float(np.mean(f1))) if f1 else 0.0)
        return res, score

    def show(res):
        return ", ".join(("%s %.1f" if k.endswith("F1") else "%s %.2f%%") % (k, v) for k, v in res.items())

    start, _ = dev_report()
    log("before fine-tuning: " + show(start))
    best, best_score, history, step = None, None, [dict(epoch=0, **start)], 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0, losses = time.time(), []
        epoch_pool = joined_pool(pool, join, np.random.default_rng(epoch))
        for idx in batches(epoch_pool, max_seconds, seed=epoch):
            wave, lengths, targets = collate(epoch_pool, idx, device, pad="train" if pad else None, seed=step)
            with torch.no_grad():
                wave = augment(wave, lengths)
            x, frames = frontend(wave, lengths)
            x = spec_augment(x, frames)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
            loss = ctc_loss(logits.float().log_softmax(1), epoch_pool, idx, targets, out_lengths(frames))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(loss.item())
            step += 1
            if max_steps and step >= max_steps:
                break
        res, score = dev_report()
        history.append(dict(epoch=epoch, loss=float(np.mean(losses)), **res))
        guard_ok = (not guard_set) or res[guard_set[0]] <= start[guard_set[0]] + guard_margin
        log("epoch %2d  loss %.3f  %s  (%.0f s)%s" % (epoch, np.mean(losses), show(res), time.time() - t0,
            "" if guard_ok else "  [guard: LibriSpeech degraded, not selectable]"))
        if guard_ok and (best_score is None or score < best_score):
            best_score = score
            best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(dict(state_dict=best, input_scale=INPUT_SCALE, epoch=epoch, dev=res,
                            tokens=TOKENS_PUNCT[:n_classes],
                            note="CynipsNet v1.0 fine-tuned; weights derived from CC BY-NC-SA 4.0 CynipsNet"),
                       out_path)
            log("  saved best -> %s" % out_path)
        if max_steps and step >= max_steps:
            break
    if best is not None:
        model.load_state_dict(best)
    return history


def unpack(archive, dest):
    """Extract a .zip or .tar(.gz) of recordings or a Common Voice release; return dest."""
    os.makedirs(dest, exist_ok=True)
    if archive.endswith(".zip"):
        zipfile.ZipFile(archive).extractall(dest)
    else:
        with tarfile.open(archive) as t:
            t.extractall(dest)
    return dest


def find_manifests(root):
    return [os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs if f == "manifest.jsonl"]

## 2. Settings

In [ ]:
DRIVE = "/content/drive/MyDrive/cynipsnet"
START_FROM = "cynipsnet_punct.pt"      # round 3; "cynipsnet_finetuned.pt" = round 1, "cynipsnet_v1_float.pt" = the original
PUNCTUATION = True           # comma, full stop, question mark
USE_SVARAH = True            # needs HF_TOKEN in the Colab secrets
COMMON_VOICE_ARCHIVE = None  # e.g. DRIVE + "/cv-corpus-en.tar.gz" from Mozilla Data Collective
CV_LIMIT = 6000              # accented clips per split at most
LIBRI_TRAIN_UTTS = 8000      # LibriSpeech train-clean-100 utterances: the punctuation teacher, and memory of clean speech
EPOCHS = 12
JOIN = 0.3                   # share of clips joined 2-3 at a time each epoch: full stops followed by speech
QUESTION_BOOST = 3.0         # questions are rare: clips with a question mark are seen this many times
LR = 1e-4
MAX_SECONDS_PER_BATCH = 240  # lower it if the GPU runs out of memory
WEIGHTS = {"board": 3.0, "pc": 1.5, "librispeech": 0.7}  # how often each source is seen per epoch
OUT = DRIVE + "/cynipsnet_punct2.pt"

## 3. Load the starting model and LibriSpeech-PC, and check the model before anything else

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import importlib, os, glob
import cynipsnet_finetune as cf
importlib.reload(cf)
device = torch.device("cuda")
model = cf.load_baseline(DRIVE + "/" + START_FROM, device)
if PUNCTUATION:
    model = cf.add_punctuation(model)
print("output classes:", model.c4.out_channels)
fe = cf.FrontEnd().to(device)

# LibriSpeech-PC: 25 MB of punctuated transcripts (no audio; the audio comes from Hugging Face)
!test -f librispeech_pc/train-clean-100.json || (mkdir -p librispeech_pc && (wget -q -O lpc.tgz https://www.openslr.org/resources/145/manifests.tar.gz || wget -q -O lpc.tgz https://openslr.elda.org/resources/145/manifests.tar.gz) && tar xzf lpc.tgz -C librispeech_pc)
pc = {split: cf.load_librispeech_pc("librispeech_pc/%s.json" % split) for split in ("train-clean-100", "dev-clean", "test-clean")}
print({k: len(v) for k, v in pc.items()}, "punctuated transcripts")

# LibriSpeech comes from its Hugging Face copy: seconds on Colab, against tens of minutes from openslr.org
libri_dev = cf.load_librispeech_hf("validation", limit=400, seed=0, punct_text=pc["dev-clean"] if PUNCTUATION else None)
w, ex = cf.evaluate(model, fe, libri_dev, device)
print("starting model, LibriSpeech dev-clean WER: %.2f%%  (expected 5-8%%)" % w, ex[:1])
assert w < 12, "This is not a CynipsNet checkpoint, or the front-end is broken: stop here."

## 4. Data

In [ ]:
import time
train, dev, test = {}, {}, {}
t0 = time.time()
step = lambda what: print("[%4.0f s] %s" % (time.time() - t0, what), flush=True)

if USE_SVARAH:
    from google.colab import userdata
    step("Svarah: downloading 1.1 GB and decoding 6656 clips (a few minutes)")
    sv = cf.load_svarah(userdata.get("HF_TOKEN"), punct=PUNCTUATION)
    train["svarah"], dev["svarah"], test["svarah"] = cf.split_by_speaker(sv, dev=0.1, test=0.2)
    step("Svarah: %d clips, %d with punctuation in their transcript" % (len(sv), sum(it.punct for it in sv)))

if COMMON_VOICE_ARCHIVE:
    step("Common Voice: unpacking and decoding the accented clips")
    cf.unpack(COMMON_VOICE_ARCHIVE, "/content/cv")
    cv_dir = next(d for d, _, fs in os.walk("/content/cv") if "train.tsv" in fs)
    cv = cf.load_common_voice(cv_dir, limit=CV_LIMIT, punct=PUNCTUATION)
    train["commonvoice"], dev["commonvoice"], test["commonvoice"] = cv.get("train", []), cv.get("dev", []), cv.get("test", [])

# Recordings from datacollect/karaoke_app.py: zips or folders under DRIVE with a manifest.jsonl
for k, archive in enumerate(sorted(glob.glob(DRIVE + "/*.zip"))):
    cf.unpack(archive, "/content/rec/%d_%s" % (k, os.path.basename(archive)[:-4]))
for mf in cf.find_manifests("/content/rec") + cf.find_manifests(DRIVE):
    items = cf.load_manifest(mf, punct=PUNCTUATION)
    for src in sorted({i.source for i in items}):
        part = [i for i in items if i.source == src]
        if "test" in mf.lower():
            test[src] = test.get(src, []) + part
        else:
            tr, dv, _ = cf.split_by_speaker(part, dev=0.1, test=0.0)
            train[src], dev[src] = train.get(src, []) + tr, dev.get(src, []) + dv

step("LibriSpeech train-clean-100: 5 of its 14 shards (2.3 GB), %d utterances used" % LIBRI_TRAIN_UTTS)
train["librispeech"] = cf.load_librispeech_hf("train.100", shards=(0, 3, 6, 9, 12), limit=LIBRI_TRAIN_UTTS, seed=1,
                                              punct_text=pc["train-clean-100"] if PUNCTUATION else None)
dev["librispeech"] = libri_dev[200:]   # selection; libri_dev[:200] guards against forgetting
test["librispeech"] = cf.load_librispeech_hf("test", limit=400, seed=2, punct_text=pc["test-clean"] if PUNCTUATION else None)
# Three sentences per clip with pauses between, as the board hears a stream: full stops must come mid-clip
import numpy as np
def joined(items, n=3, seed=0):
    rng = np.random.default_rng(seed)
    return [cf.join_clips(items[k:k + n], rng) for k in range(0, len(items) - n + 1, n)]
dev["librispeech joined"] = joined(libri_dev[200:])
test["librispeech joined"] = joined(test["librispeech"], seed=1)
step("LibriSpeech: %d training utterances" % len(train["librispeech"]))

for name, sets in (("train", train), ("dev", dev), ("test", test)):
    for k, v in sets.items():
        if v:
            print(cf.describe("%s/%s" % (name, k), v), "| punctuated: %d" % sum(it.punct for it in v))
train = {k: v for k, v in train.items() if v}
dev = {k: v for k, v in dev.items() if v}
test = {k: v for k, v in test.items() if v}

## 5. Starting accuracy on every test set

In [ ]:
def report(model):
    out = {}
    for k, v in test.items():
        w, pairs = cf.evaluate(model, fe, v, device)
        marked = [p for p, it in zip(pairs, v) if it.punct]
        out[k] = dict(wer=round(w, 2), marks=cf.punct_report(marked) if marked and PUNCTUATION else None)
    return out

def show(rep):
    for k, r in rep.items():
        marks = r["marks"]
        print("%-12s WER %6.2f%%" % (k, r["wer"]) + ("   marks F1 %.1f  (, %.1f  . %.1f  ? %.1f)" % (
            marks["all"][2], marks[","][2], marks["."][2], marks["?"][2]) if marks else ""))

baseline = report(model)
show(baseline)

## 6. Fine-tune (the best epoch is saved to Drive after every epoch)

In [ ]:
history = cf.train(model, fe, train, dev, device, epochs=EPOCHS, lr=LR, max_seconds=MAX_SECONDS_PER_BATCH,
                   weights=WEIGHTS, guard_set=("librispeech guard", libri_dev[:200]), out_path=OUT,
                   join=JOIN, question_boost=QUESTION_BOOST)

## 7. Before / after on held-out test sets, and examples

In [ ]:
import json
best = torch.load(OUT, weights_only=False)
model.load_state_dict(best["state_dict"])
after = report(model)
print("before:"); show(baseline)
print("after (epoch %d):" % best["epoch"]); show(after)
for k, v in test.items():
    _, pairs = cf.evaluate(model, fe, v[:40], device)
    print("\n%s examples:" % k)
    for ref, hyp in pairs[:5]:
        print("  REF", ref)
        print("  HYP", hyp)
json.dump(dict(baseline=baseline, after=after, history=history, epoch=best["epoch"], start_from=START_FROM),
          open(DRIVE + "/finetune_report_punct2.json", "w"), indent=1)
print("\nCheckpoint:", OUT)

## 8. Put it on the board (on your PC)
Download `cynipsnet_punct2.pt` and `finetune_report_punct2.json` from Drive into `cynipsnet_re/`, then:
```
python deploy/deploy.py cynipsnet_punct2.pt --name cynipsnet_punct2
```
It quantizes to int8, reports the int8 WER and the punctuation scores through the firmware's streaming
path, compiles with ST Edge AI (32 output classes), builds and signs the firmware (sentence case on the
screen), and prints the flash command (bootloader, app **and** weights together).